# 0️⃣4️⃣ - Managing your Admin Users
![Persona](https://img.shields.io/badge/%F0%9F%91%A4%20Persona-%F0%9F%9B%A0%EF%B8%8F%20Platform%20%2F%20IT%20Administrator-lightgrey) ![Difficulty](https://img.shields.io/badge/%F0%9F%9A%A6%20Difficulty-Beginner-green) ![RADKit version](https://img.shields.io/badge/RADKit-1.9.6-blue?logo=cisco&logoColor=white) ![Python version](https://img.shields.io/badge/Python-3.12%2B-yellow?logo=python&logoColor=white)

> In this playbook we will explore how to list, create, update, and delete admin accounts on the service.


### 🛠️ Before You Begin

If you haven't set up your environment yet, follow the instructions in [SETUP.md](../SETUP.md) before running any cells.

This playbook builds on concepts from:
- [01 — Managing your Remote Users](./01-manage-users.ipynb)
- [02 — Managing your Devices](./02-manage-devices.ipynb)
- [03 — Managing your Labels](./03-manage-labels.ipynb)


---

### 📋 What you'll learn

| # | Topic |
|---|-------|
| 1 | How to list all admin users on a RADKit service |
| 2 | How to retrieve details of a specific admin account |
| 3 | How to create a new admin account with a specified role |
| 4 | How to update an existing admin's details |
| 5 | How to delete an admin account |


---

## 🔍 Method 1: List Admin Users

**Best for:** Auditing the current admin roster, verifying account status, or looking up details of a specific admin.

**How it works:**
1. Connect to the service with `ControlAPI.create()`.
2. Call `list_admins()` to retrieve all configured admin accounts.
3. Use `get_admin(username)` to fetch details of a specific admin.

**What you need:**
- The RADKit server address and superadmin password

### 1.1 List all admins and retrieve a specific one


In [ ]:
from getpass import getpass
from radkit_service.control_api import ControlAPI

radkit_server = input("👤 Enter your RADKit server IP address: ")
radkit_password = getpass("🔑 Please input the password of your RADKit server: ")

with ControlAPI.create(
    base_url=f"https://{radkit_server}:8081/api/v1",
    admin_name="superadmin",
    admin_password=radkit_password,
    http_client_kwargs=dict(verify=False),
) as service:

    print("👑 Superusers on this RADKit server:")
    for superuser in service.list_admins().root.result:
        username = superuser.username
        masked = username[:2] + "*" * (len(username) - 2) if len(username) > 2 else "**"
        enabled_icon = "✅" if superuser.enabled else ""
        print(f"👑 Username: {masked} {enabled_icon}".strip())

    specific_admin = service.get_admin("superadmin").root.result[0]
    print(f"\n🔍 Details of 'superadmin':"
          f"\n👑 Username: {specific_admin.username}"
          f"\n🔒 Enabled: {'Yes' if specific_admin.enabled else 'No'}"
          f"\n📧 Email: {specific_admin.email or 'N/A'}")


👑 Superusers on this RADKit server:
👑 Username: su******** ✅
👑 Username: ls****** ✅
👑 Username: ad*** ✅

🔍 Details of 'superadmin':
👑 Username: superadmin
🔒 Enabled: Yes
📧 Email: N/A


---

## ➕ Method 2: Create an Admin User

**Best for:** Provisioning new admin accounts for team members or automation systems that need access to the RADKit service.

**How it works:**
1. Connect to the service with `ControlAPI.create()`.
2. Call `create_admin()` with the desired username, password, role, and optional metadata.
3. Confirm creation by retrieving the new admin with `get_admin()`.

**What you need:**
- The RADKit server address and superadmin password
- A strong password aligned with RADKit password policies

> If `initial` is `True`, this request is unauthenticated and only works when there are no existing admin users in the system.

The following two roles are enabled by default:

**`basic-admin`**
- Modify Devices
- Read Device Templates
- Read Activity
- Read Roles
- Read External Sources
- Modify Labels
- Read Logs
- Read Settings
- Modify Remote Users

**`sysadmin`**
- Full access to all system features

### 2.1 Create a new admin and confirm the result


In [ ]:
from getpass import getpass
from radkit_service.control_api import ControlAPI

radkit_server = input("👤 Enter your RADKit server IP address: ")
radkit_password = getpass("🔑 Please input the password of your RADKit server: ")

with ControlAPI.create(
    base_url=f"https://{radkit_server}:8081/api/v1",
    admin_name="superadmin",
    admin_password=radkit_password,
    http_client_kwargs=dict(verify=False),
) as service:
    result = service.create_admin(
        enabled=False,
        admin_name="testadmin",
        password="TestAdminPass123",
        email="admin@acme.com",
        full_name="Test Admin",
        description="This is a test admin account created for demonstration purposes.",
        role="basic-admin"
    )

    specific_admin = service.get_admin("testadmin").root.result[0]
    print(f"\n🔍 Details of 'testadmin':"
          f"\n👑 Username: {specific_admin.username}"
          f"\n🔒 Enabled: {'Yes' if specific_admin.enabled else 'No'}"
          f"\n📧 Email: {specific_admin.email or 'N/A'}")



🔍 Details of 'testadmin':
👑 Username: testadmin
🔒 Enabled: No
📧 Email: admin@acme.com


---

## ✏️ Method 3: Update an Admin User

**Best for:** Modifying account metadata such as email, full name, or enabling/disabling an admin account.

**How it works:**
1. Connect to the service with `ControlAPI.create()`.
2. Call `update_admin()` with the target `admin_name` and the fields to change.
3. Confirm the update by retrieving the admin with `get_admin()`.

> It is **NOT** possible to update the admin password using `update_admin()`.

**What you need:**
- The RADKit server address and superadmin password
- The username of the admin to update

### 3.1 Update an admin's details and confirm the result


In [ ]:
from getpass import getpass
from radkit_service.control_api import ControlAPI

radkit_server = input("👤 Enter your RADKit server IP address: ")
radkit_password = getpass("🔑 Please input the password of your RADKit server: ")

with ControlAPI.create(
    base_url=f"https://{radkit_server}:8081/api/v1",
    admin_name="superadmin",
    admin_password=radkit_password,
    http_client_kwargs=dict(verify=False),
) as service:
    result = service.update_admin(
        enabled=True,
        admin_name="testadmin",
        email="test2@acme.com"
    )

    specific_admin = service.get_admin("testadmin").root.result[0]
    print(f"\n🔍 Details of 'testadmin':"
          f"\n👑 Username: {specific_admin.username}"
          f"\n🔒 Enabled: {'Yes' if specific_admin.enabled else 'No'}"
          f"\n📧 Email: {specific_admin.email or 'N/A'}")



🔍 Details of 'testadmin':
👑 Username: testadmin
🔒 Enabled: Yes
📧 Email: test2@acme.com


---

## 🗑️ Method 4: Delete an Admin User

**Best for:** Removing stale, test, or decommissioned admin accounts to keep the admin roster clean.

**How it works:**
1. Connect to the service with `ControlAPI.create()`.
2. Call `delete_admin()` with the target `admin_name`.
3. Confirm deletion by listing all remaining admins with `list_admins()`.

**What you need:**
- The RADKit server address and superadmin password
- The username of the admin to delete

### 4.1 Delete an admin and verify removal


In [ ]:
from getpass import getpass
from radkit_service.control_api import ControlAPI

radkit_server = input("👤 Enter your RADKit server IP address: ")
radkit_password = getpass("🔑 Please input the password of your RADKit server: ")

with ControlAPI.create(
    base_url=f"https://{radkit_server}:8081/api/v1",
    admin_name="superadmin",
    admin_password=radkit_password,
    http_client_kwargs=dict(verify=False),
) as service:
    result = service.delete_admin(
        admin_name="testadmin"
    )

    print("👑 Superusers on this RADKit server:")
    for superuser in service.list_admins().root.result:
        username = superuser.username
        masked = username[:2] + "*" * (len(username) - 2) if len(username) > 2 else "**"
        enabled_icon = "✅" if superuser.enabled else ""
        print(f"👑 Username: {masked} {enabled_icon}".strip())


👑 Superusers on this RADKit server:
👑 Username: su******** ✅
👑 Username: ls****** ✅
👑 Username: ad*** ✅


---

## 🔀 Which Method Should I Use?

| Dimension | Method 1: List | Method 2: Create | Method 3: Update | Method 4: Delete |
|-----------|---------------|-----------------|-----------------|-----------------|
| **Goal** | Audit / discover admins | Provision new admin | Modify existing admin | Remove admin |
| **Auth required** | Yes (superadmin) | Yes (or `initial=True`) | Yes (superadmin) | Yes (superadmin) |
| **Reversible** | Read-only | Yes (delete later) | Yes (update again) | No |
| **Best for** | Auditing access | Onboarding team members | Updating email or status | Cleanup |
